In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz, load_npz
import pickle
import os

In [2]:
INPUT_FILES = {
    "fully_cleaned": "../schemas/fully_cleaned.csv",
    "without_lemmatization": "../schemas/without_lemmatization.csv",
    "without_stopwords": "../schemas/without_stopwords_lowercasing_punct.csv"
}

OUTPUT_FILES = {
    "fully_cleaned": "text_representation_cleaned_tfidf",
    "without_lemmatization": "text_representation_tfidf_without_Lemmatization",
    "without_stopwords": "text_representation_tfidf_withoutStopwords_RemovePunctuation"
}

TEXT_COLUMN = "cleaned_review"  # change if needed
MAX_FEATURES = 5000

In [3]:
def process_file(input_path, prefix):
    print(f"\nProcessing: {input_path}")

    df = pd.read_csv(input_path)
    df = df.dropna(subset=[TEXT_COLUMN])

    texts = df[TEXT_COLUMN].astype(str)

    # TF-IDF
    vectorizer = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.8
    )

    X = vectorizer.fit_transform(texts)

    # -----------------------------
    # Save sparse matrix
    # -----------------------------
    save_npz(f"{prefix}.npz", X)

    # -----------------------------
    # Save vectorizer
    # -----------------------------
    with open(f"{prefix}_vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)

    print(f"Saved: {prefix}.npz")
    print(f"Shape: {X.shape}")

In [4]:
for key in INPUT_FILES:
    process_file(INPUT_FILES[key], "./TF_IDF/"+ OUTPUT_FILES[key])

print("\nAll TF-IDF files generated.")


Processing: ../schemas/fully_cleaned.csv
Saved: ./TF_IDF/text_representation_cleaned_tfidf.npz
Shape: (4996, 5000)

Processing: ../schemas/without_lemmatization.csv
Saved: ./TF_IDF/text_representation_tfidf_without_Lemmatization.npz
Shape: (4996, 5000)

Processing: ../schemas/without_stopwords_lowercasing_punct.csv
Saved: ./TF_IDF/text_representation_tfidf_withoutStopwords_RemovePunctuation.npz
Shape: (4998, 5000)

All TF-IDF files generated.


In [5]:
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec
import os

In [6]:
INPUT_FILES = {
    "fully_cleaned": "../schemas/fully_cleaned.csv",
    "without_lemmatization": "../schemas/without_lemmatization.csv",
    "without_stopwords": "../schemas/without_stopwords_lowercasing_punct.csv"
}

OUTPUT_FILES = {
    "fully_cleaned": "text_representation_glove_cleaned",
    "without_lemmatization": "text_representation_glove_without_Lemmatization",
    "without_stopwords": "text_representation_glove_withoutStopwords_removePunctuation"
}

TEXT_COLUMN = "cleaned_review"

GLOVE_PATH = "glove.6B.100d.txt"

In [7]:
model = KeyedVectors.load_word2vec_format(GLOVE_PATH, binary=False, no_header=True)

EMBEDDING_DIM = model.vector_size

In [8]:
# -----------------------------
# Helper: average word vectors
# -----------------------------
def document_vector(text):
    words = text.split()

    vectors = [model[word] for word in words if word in model]

    if len(vectors) == 0:
        return np.zeros(EMBEDDING_DIM)

    return np.mean(vectors, axis=0)

In [9]:
def process_file(input_path, prefix):
    print(f"\nProcessing: {input_path}")

    df = pd.read_csv(input_path)
    df = df.dropna(subset=[TEXT_COLUMN])

    texts = df[TEXT_COLUMN].astype(str)

    # Compute embeddings
    embeddings = np.vstack([document_vector(text) for text in texts])

    # Convert to DataFrame
    cols = [f"dim_{i}" for i in range(EMBEDDING_DIM)]
    emb_df = pd.DataFrame(embeddings, columns=cols)

    # Save
    output_path = f"./Glove/{prefix}.csv"
    emb_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print(f"Shape: {emb_df.shape}")

In [10]:
for key in INPUT_FILES:
    process_file(INPUT_FILES[key], OUTPUT_FILES[key])

print("\nAll GloVe representations generated.")


Processing: ../schemas/fully_cleaned.csv
Saved: ./Glove/text_representation_glove_cleaned.csv
Shape: (4996, 100)

Processing: ../schemas/without_lemmatization.csv
Saved: ./Glove/text_representation_glove_without_Lemmatization.csv
Shape: (4996, 100)

Processing: ../schemas/without_stopwords_lowercasing_punct.csv
Saved: ./Glove/text_representation_glove_withoutStopwords_removePunctuation.csv
Shape: (4998, 100)

All GloVe representations generated.


In [11]:
X = load_npz("../text_representation/TF_IDF/text_representation_cleaned_tfidf.npz")
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 421064 stored elements and shape (4996, 5000)>